In [1]:
cd drive/My Drive/quant_machine

/content/drive/My Drive/quant_machine


In [2]:
from finterstellar import util
import pandas as pd
import numpy as np
import pandas_datareader.data as web

In [3]:
df = web.DataReader('AAPL', 'yahoo', start='2019-01-01')['Adj Close'].round(2)
df.tail()

Date
2020-11-25    116.03
2020-11-27    116.59
2020-11-30    119.05
2020-12-01    122.72
2020-12-02    123.08
Name: Adj Close, dtype: float64

In [4]:
def get_price(symbol, start_date=None, end_date=None):
    symbol = util.str_to_list(symbol)
    end_date = pd.to_datetime(end_date).date() if end_date else pd.Timestamp.today().date()
    start_date = pd.to_datetime(start_date).date() if start_date else util.months_before(end_date, 12)
    df = web.DataReader(symbol, 'yahoo', start=start_date, end=end_date)['Adj Close'].round(2)
    return df

In [5]:
symbol = 'M'
df = get_price(symbol)
df.tail(10)

Symbols,M
Date,
2020-11-18,8.99
2020-11-19,9.18
2020-11-20,9.05
2020-11-23,10.41
2020-11-24,10.86
2020-11-25,11.00
2020-11-27,10.85
2020-11-30,10.21
2020-12-01,10.40


In [9]:
def get_rsi(price_df, w=5):
    df = pd.DataFrame()
    df['price'] = price_df
    df.fillna(method='ffill', inplace=True)   # 들어온 데이터의 구멍을 메꿔준다
    if len(df) > w+1:
        df['diff'] = df['price'].diff()   # 일별 가격차이 계산
        df['diff_abs'] = df['price'].diff().abs()   # 일별 가격차이의 절대값 계산
        df['diff_positive'] = df['diff'].mask(df['diff']<0, 0)   # 가격차이가 +인 경우만 발라내기
        df['diff_abs_rolling_sum'] = df['diff_abs'].rolling(w).sum()   # RSI 분모
        df['diff_positive_rolling_sum'] = df['diff_positive'].rolling(w).sum()   # RSI 분자
        df['RSI'] = df['diff_positive_rolling_sum'].truediv(df['diff_abs_rolling_sum'], fill_value=.5)*100   # RSI
        return df['RSI'].round(2)
    else:
        return None

In [10]:
rsi = get_rsi(df[symbol], w=5)
rsi.tail()

Date
2020-11-25    94.27
2020-11-27    87.44
2020-11-30    71.17
2020-12-01    49.68
2020-12-02    52.98
Name: RSI, dtype: float64

In [11]:
# df = pd.concat([df, rsi], axis=1)
df['RSI'] = get_rsi(df[symbol], w=5)
df.head(20)

Symbols,M,알에스아이
Date,,
2019-12-03,13.93,NaN
2019-12-04,13.91,NaN
2019-12-05,14.16,NaN
2019-12-06,14.20,NaN
2019-12-09,14.51,NaN
2019-12-10,14.75,97.67
2019-12-11,14.71,95.45
2019-12-12,15.08,96.00
2019-12-13,14.58,63.01


In [ ]:
# 한글세팅
# !apt-get install -y -qq fonts-nanum

# from matplotlib import font_manager
# font_manager._rebuild()
# set(sorted([f.name for f in font_manager.fontManager.ttflist]))

In [12]:
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
ScalarFormatter().set_scientific(False)
font = 'NanumSquareRound, AppleGothic, Malgun Gothic, DejaVu Sans'
plt.style.use('seaborn')
plt.rcParams['font.family'] = font
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['lines.linewidth'] = 1.5
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['grid.alpha'] = 0.7
plt.rcParams['lines.antialiased'] = True
plt.rcParams['figure.figsize'] = [10.0, 5.0]
plt.rcParams['savefig.dpi'] = 96
plt.rcParams['font.size'] = 12
plt.rcParams['legend.fontsize'] = 'medium'
plt.rcParams['figure.titlesize'] = 'medium'

In [2]:
def draw_chart(df, left, right=None, log=False):
    left = util.str_to_list(left)
    fig, ax1 = plt.subplots()
    x = df.index
    i = 1
    for c in left:
        ax1.plot(x, df[c], label=c, color='C'+str(i), alpha=1)
        i += 1
    if log:
        ax1.set_yscale('log')
        ax1.yaxis.set_major_formatter(ScalarFormatter())
        ax1.yaxis.set_minor_formatter(ScalarFormatter())
    # secondary y
    if right is not None:
        right = util.str_to_list(right)
        ax2 = ax1.twinx()
        i = 6
        for c in right:
            ax2.plot(x, df[c], label=c+'(R)', color='C'+str(i), alpha=0.7)
            ax1.plot(np.nan, label=c+'(R)', color='C'+str(i))
            i += 1
        ax2.grid(False)
        if log:
            ax2.set_yscale('log')
            ax2.yaxis.set_major_formatter(ScalarFormatter())
            ax2.yaxis.set_minor_formatter(ScalarFormatter())
    ax1.legend(loc=0)

In [3]:
draw_chart(df, 'M', '알에스아이')

NameError: ignored

In [ ]:
def get_rsi(price_df, w=5):
    df = pd.DataFrame()
    df['price'] = price_df
    df.fillna(method='ffill', inplace=True)   # 들어온 데이터의 구멍을 메꿔준다
    if len(df) > w+1:
        df['diff'] = df['price'].diff()   # 일별 가격차이 계산
        df['diff_abs'] = df['price'].diff().abs()   # 일별 가격차이의 절대값 계산
        df['diff_positive'] = df['diff'].mask(df['diff']<0, 0)   # 가격차이가 +인 경우만 발라내기
        df['diff_abs_rolling_mean'] = df['diff_abs'].rolling(w).mean()   # RSI 분모
        df['diff_positive_rolling_mean'] = df['diff_positive'].rolling(w).mean()   # RSI 분자
        df['rsi'] = df['diff_positive_rolling_mean'].truediv(df['diff_abs_rolling_mean'], fill_value=.5)*100   # RSI
        return df['rsi'].round(2)
    else:
        return None

In [16]:
get_rsi(df[symbol], w=5).tail()

Date
2020-11-25    94.27
2020-11-27    87.44
2020-11-30    71.17
2020-12-01    49.68
2020-12-02    52.98
Name: rsi, dtype: float64

In [17]:
def get_wrsi(price_df, w=5):
    df = pd.DataFrame()
    df['price'] = price_df
    df.fillna(method='ffill', inplace=True)   # 들어온 데이터의 구멍을 메꿔준다
    if len(df) > w+1:
        df['diff'] = df['price'].diff()   # 일별 가격차이 계산
        df['diff_abs'] = df['price'].diff().abs()   # 일별 가격차이의 절대값 계산
        df['diff_positive'] = df['diff'].mask(df['diff']<0, 0)   # 가격차이가 +인 경우만 발라내기
        weight = pd.array(range(1, w+1))
        df['weighted_diff_abs_rolling_sum'] = df['diff_abs'].rolling(w).apply(lambda x: np.dot(x,weight))   # WRSI 분모
        df['weighted_diff_positive_rolling_sum'] = df['diff_positive'].rolling(w).apply(lambda x: np.dot(x,weight))   # WRSI 분자
        df['wrsi'] = df['weighted_diff_positive_rolling_sum'].truediv(df['weighted_diff_abs_rolling_sum'], fill_value=.5)*100   # WRSI 
        return df['wrsi'].round(2)
    else:
        return None

In [18]:
df['wrsi'] = get_wrsi(df[symbol], w=5)
df.tail(3)

Symbols,M,알에스아이,wrsi
Date,,,
2020-11-30,10.21,71.17,41.36
2020-12-01,10.40,49.68,35.82
2020-12-02,10.96,52.98,62.50


In [19]:
df['ma'] = df[symbol].rolling(5).mean()
df['rolling_sum'] = df[symbol].rolling(5).sum()
df.tail()

Symbols,M,알에스아이,wrsi,ma,rolling_sum
Date,,,,,
2020-11-25,11.00,94.27,96.30,10.100,50.50
2020-11-27,10.85,87.44,84.03,10.434,52.17
2020-11-30,10.21,71.17,41.36,10.666,53.33
2020-12-01,10.40,49.68,35.82,10.664,53.32
2020-12-02,10.96,52.98,62.50,10.684,53.42


In [20]:
def get_index(price_df):
    df = pd.DataFrame()
    df['price'] = price_df
    df['idx'] = (df['price'] / df['price'][0] * 100).round(2)
    return df['idx']

In [21]:
df['idx'] = get_index(df[symbol])
df.tail(3)

Symbols,M,알에스아이,wrsi,ma,rolling_sum,idx
Date,,,,,,
2020-11-30,10.21,71.17,41.36,10.666,53.33,73.30
2020-12-01,10.40,49.68,35.82,10.664,53.32,74.66
2020-12-02,10.96,52.98,62.50,10.684,53.42,78.68


In [22]:
df['daily_change'] = df[symbol].diff()
df['daily_change_rate'] = df[symbol].pct_change()
df.tail()

Symbols,M,알에스아이,wrsi,ma,rolling_sum,idx,daily_change,daily_change_rate
Date,,,,,,,,
2020-11-25,11.00,94.27,96.30,10.100,50.50,78.97,0.14,0.012891
2020-11-27,10.85,87.44,84.03,10.434,52.17,77.89,-0.15,-0.013636
2020-11-30,10.21,71.17,41.36,10.666,53.33,73.30,-0.64,-0.058986
2020-12-01,10.40,49.68,35.82,10.664,53.32,74.66,0.19,0.018609
2020-12-02,10.96,52.98,62.50,10.684,53.42,78.68,0.56,0.053846


In [23]:
df['sigma'] = df[symbol].rolling(5).std()
df

Symbols,M,알에스아이,wrsi,ma,rolling_sum,idx,daily_change,daily_change_rate,sigma
Date,,,,,,,,,
2019-12-03,13.93,NaN,NaN,NaN,NaN,100.00,NaN,NaN,NaN
2019-12-04,13.91,NaN,NaN,NaN,NaN,99.86,-0.02,-0.001436,NaN
2019-12-05,14.16,NaN,NaN,NaN,NaN,101.65,0.25,0.017973,NaN
2019-12-06,14.20,NaN,NaN,NaN,NaN,101.94,0.04,0.002825,NaN
2019-12-09,14.51,NaN,NaN,14.142,70.71,104.16,0.31,0.021831,0.243865
...,...,...,...,...,...,...,...,...,...
2020-11-25,11.00,94.27,96.30,10.100,50.50,78.97,0.14,0.012891,0.926364
2020-11-27,10.85,87.44,84.03,10.434,52.17,77.89,-0.15,-0.013636,0.804817
2020-11-30,10.21,71.17,41.36,10.666,53.33,73.30,-0.64,-0.058986,0.337831


In [24]:
df['z_score'] = (df[symbol] - df['ma']) / df['sigma']
df.tail()

Symbols,M,알에스아이,wrsi,ma,rolling_sum,idx,daily_change,daily_change_rate,sigma,z_score
Date,,,,,,,,,,
2020-11-25,11.00,94.27,96.30,10.100,50.50,78.97,0.14,0.012891,0.926364,0.971540
2020-11-27,10.85,87.44,84.03,10.434,52.17,77.89,-0.15,-0.013636,0.804817,0.516888
2020-11-30,10.21,71.17,41.36,10.666,53.33,73.30,-0.64,-0.058986,0.337831,-1.349786
2020-12-01,10.40,49.68,35.82,10.664,53.32,74.66,0.19,0.018609,0.339750,-0.777042
2020-12-02,10.96,52.98,62.50,10.684,53.42,78.68,0.56,0.053846,0.356693,0.773774


In [25]:
def get_period(df):
    df.dropna(inplace=True)
    end_date = df.index[-1]
    start_date = df.index[0]
    days_between = (end_date - start_date).days
    return abs(days_between)

In [26]:
def annualize(rate, period):
    if period < 365:
        rate = ((rate-1) / period * 365) + 1
    elif period > 365:
        rate = rate ** (365 / period)
    return round(rate, 2)

In [27]:
annualize(1.1, 91)

1.4

In [28]:
df.head()

Symbols,M,알에스아이,wrsi,ma,rolling_sum,idx,daily_change,daily_change_rate,sigma,z_score
Date,,,,,,,,,,
2019-12-03,13.93,NaN,NaN,NaN,NaN,100.00,NaN,NaN,NaN,NaN
2019-12-04,13.91,NaN,NaN,NaN,NaN,99.86,-0.02,-0.001436,NaN,NaN
2019-12-05,14.16,NaN,NaN,NaN,NaN,101.65,0.25,0.017973,NaN,NaN
2019-12-06,14.20,NaN,NaN,NaN,NaN,101.94,0.04,0.002825,NaN,NaN
2019-12-09,14.51,NaN,NaN,14.142,70.71,104.16,0.31,0.021831,0.243865,1.509033


In [29]:
def get_trade_signal(df, factor, buy, sell):
    df['trade'] = np.nan
    if buy > sell:
        df['trade'].mask(df[factor]>buy, 'buy', inplace=True)
        df['trade'].mask(df[factor]<sell, 'zero', inplace=True)
    else:
        df['trade'].mask(df[factor]<buy, 'buy', inplace=True)
        df['trade'].mask(df[factor]>sell, 'zero', inplace=True)
    df['trade'].fillna(method='ffill', inplace=True)
    df['trade'].fillna('zero', inplace=True)
    return df['trade']

In [30]:
df['trade'] = get_trade_signal(df, factor='rsi', buy=30, sell=60)

KeyError: ignored

In [ ]:
def get_position(df):
    df['position'] = ''
    df['position'].mask((df['trade'].shift(1)=='zero') & (df['trade']=='zero'), 'zz', inplace=True)
    df['position'].mask((df['trade'].shift(1)=='zero') & (df['trade']=='buy'), 'zl', inplace=True)
    df['position'].mask((df['trade'].shift(1)=='buy') & (df['trade']=='zero'), 'lz', inplace=True)
    df['position'].mask((df['trade'].shift(1)=='buy') & (df['trade']=='buy'), 'll', inplace=True)
    return df['position']

In [ ]:
df['position'] = get_position(df)

In [ ]:
def trade(df, cost=.001):
    df['signal_price'] = np.nan
    df['signal_price'].mask(df['position']=='zl', df.iloc[:,0], inplace=True)
    df['signal_price'].mask(df['position']=='lz', df.iloc[:,0], inplace=True)
    # record = df[['position','signal_price']].dropna()
    # record['rtn'] = 1
    # record['rtn'].mask(record['position']=='lz', (record['signal_price']*(1-cost))/record['signal_price'].shift(1), inplace=True)
    # record['acc_rtn'] = record['rtn'].cumprod()
    df['signal_price'].mask(df['position']=='ll', df.iloc[:,0], inplace=True)
    # df['rtn'] = record['rtn']
    # df['rtn'].fillna(1, inplace=True)
    df['daily_rtn'] = 1
    df['daily_rtn'].mask(df['position'] == 'll', df['signal_price'] / df['signal_price'].shift(1), inplace=True)
    df['daily_rtn'].mask(df['position'] == 'lz', (df['signal_price']*(1-cost)) / df['signal_price'].shift(1), inplace=True)
    df['daily_rtn'].fillna(1, inplace=True)
    df['acc_rtn'] = df['daily_rtn'].cumprod()
    df['mdd'] = (df['acc_rtn'] / df['acc_rtn'].cummax()).round(4)
    df['bm_mdd'] = (df.iloc[:, 0] / df.iloc[:, 0].cummax()).round(4)
    df.drop(columns='signal_price', inplace=True)
    return df
    return df[['rtn', 'acc_rtn']]

In [ ]:
rst = trade(df, cost=.001)
rst[-25:-10]

In [ ]:
rst = trade(df, cost=.1)
rst[-25:-10]

In [ ]:
def trade(df, cost=.001):
    df['signal_price'] = np.nan
    df['signal_price'].mask(df['position']=='zl', df[symbol], inplace=True)
    df['signal_price'].mask(df['position']=='lz', df[symbol], inplace=True)
    record = df[['position','signal_price']].dropna()
    record['rtn'] = 1
    record['rtn'].mask(record['position']=='lz', (record['signal_price']*(1-cost))/record['signal_price'].shift(1), inplace=True)
    record['acc_rtn'] = record['rtn'].cumprod()
    return record[['rtn', 'acc_rtn']]

In [ ]:
df['rtn'] = trade(df, cost=.1)['rtn']
df['acc_rtn'] = trade(df, cost=.1)['acc_rtn']
df['rtn'].fillna(1, inplace=True)
df['acc_rtn'].fillna(method='ffill', inplace=True)
df[-20:]

In [ ]:
def get_sharpe_ratio(df, rf_rate):
    rf_rate = rf_rate / 365 + 1
    sharpe_ratio = (df['daily_rtn']-rf_rate).mean() / (df['daily_rtn']-rf_rate).std()
    return round(sharpe_ratio, 2)

In [ ]:
def evaluate(df, rf_rate, cost=.001):
    df['signal_price'] = np.nan
    df['signal_price'].mask(df['position']=='zl', df[symbol], inplace=True)
    df['signal_price'].mask(df['position']=='ll', df[symbol], inplace=True)
    df['signal_price'].mask(df['position']=='lz', df[symbol], inplace=True)
    df['daily_rtn'] = 1
    # df['daily_rtn'].mask((df['position']=='ll')|(df['position']=='lz'), df['signal_price']/df['signal_price'].shift(1), inplace=True)
    df['daily_rtn'].mask(df['position'] == 'll', df['signal_price'] / df['signal_price'].shift(1), inplace=True)
    df['daily_rtn'].mask(df['position'] == 'lz', (df['signal_price']*(1-cost)) / df['signal_price'].shift(1), inplace=True)
    df['daily_rtn'].fillna(1, inplace=True)

    rst = {}
    rst['no_trades'] = (df['position']=='lz').sum()
    rst['no_win'] = (df['rtn']>1).sum()
    rst['acc_rtn'] = df['acc_rtn'][-1].round(2)
    rst['hit_ratio'] = round((df['rtn']>1).sum() / rst['no_trades'], 2)
    rst['avg_rtn'] = round(df[df['rtn']>1]['rtn'].mean(), 2)
    rst['period'] = get_period(df)
    rst['annual_rtn'] = annualize(rst['acc_rtn'], rst['period'])
    rst['sharpe_ratio'] = get_sharpe_ratio(df, rf_rate)
    rst['transaction_cost'] = cost
    return rst

In [ ]:
evaluate(df, rf_rate=.01, cost=.1)

In [ ]:
df[df['rtn']>1]['rtn'].mean()

In [ ]:
get_sharpe_ratio(df, rf_rate=.01)

In [ ]:

df['signal_price'] = np.nan
df['signal_price'].mask(df['position']=='zl', df[symbol], inplace=True)
df['signal_price'].mask(df['position']=='lz', df[symbol], inplace=True)
record = df[['position','signal_price']].dropna()
record['rtn'] = 1
record['rtn'].mask(record['position']=='lz', record['signal_price']/record['signal_price'].shift(1), inplace=True)
record['acc_rtn'] = record['rtn'].cumprod()
record

In [ ]:
record['daily_rtn'] = 1
record['daily_rtn'] = record['daily_rtn']*record['rtn']

In [ ]:
record['date'] = record.index
record['period'] = np.nan
record['period'].mask(record['position']=='lz', record['date'].diff() , inplace=True)
record

In [ ]:
record['period'].sum().days

In [ ]:
record['rtn'].mean()